# Association of the GCH1 p.Ser80Asn variant with Parkinson’s disease in East Asian populations

## Set path

In [ ]:
# Use the os package to interact with the environment
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import glob
import sys as sys

In [ ]:
# Use the os package to interact with the environment
import os

# Bring in Pandas for Dataframe functionality
import pandas as pd

import subprocess

# Numpy for basics
import numpy as np

# Use pathlib for file path manipulation
import pathlib

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.core.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

#Import Sys
import sys as sys

import re

In [ ]:
# show all columns when printing
pd.set_option('display.max_columns', None) 

In [ ]:
%%capture
%%bash

# Install plink 1.9
cd /home/jupyter/
if test -e /home/jupyter/plink; then
    echo "Plink is already installed in /home/jupyter/"
else
    echo "Plink is not installed"
    cd /home/jupyter

    wget http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 

    unzip -o plink_linux_x86_64_20190304.zip
    mv plink plink1.9
fi

In [ ]:
%%bash

# chmod plink 1.9 to make sure you have permission to run the program
chmod u+x /home/jupyter/plink1.9

In [ ]:
%%capture
%%bash

# Install plink 2.0
cd /home/jupyter/
if test -e /home/jupyter/plink2; then

echo "Plink2 is already installed in /home/jupyter/"
else
echo "Plink2 is not installed"
cd /home/jupyter/

wget https://s3.amazonaws.com/plink2-assets/alpha6/plink2_linux_x86_64_20250122.zip

unzip -o plink2_linux_x86_64_20250122.zip

fi

# chmod plink 2 to make sure you have permission to run the program
chmod u+x /home/jupyter/plink2

In [ ]:
#%%capture
#%%bash

# create target directory
OUTDIR="/home/jupyter/workspace/beagle"
mkdir -p "$OUTDIR"

# path to release 11 directory
REL11_PATH = "/path/to/release"

# download hap-ibd
wget -P "$OUTDIR" https://faculty.washington.edu/browning/hap-ibd.jar

# download beagle 5.4
wget -P "$OUTDIR" https://faculty.washington.edu/browning/beagle/beagle.29Oct24.c8e.jar

# download human genetic map
wget -P "$OUTDIR" https://bochet.gcc.biostat.washington.edu/beagle/genetic_maps/plink.GRCh38.map.zip

# unzip into same directory
unzip "$OUTDIR/plink.GRCh38.map.zip" -d "$OUTDIR"


## Preparation of covariate file

In [ ]:
key = pd.read_csv(f"{REL11_PATH}/clinical_data/master_key_release11_final_vwb.csv", low_memory=False)
key

In [ ]:
#To subset master key to keep only a few columns 
key = key[['GP2ID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'nba_label','wgs_label']]
# Renaming the columns
key.rename(columns = {'GP2ID':'IID',
                                     'baseline_GP2_phenotype_for_qc':'phenotype',
                                     'biological_sex_for_qc':'SEX', 
                                     'age_at_sample_collection':'AGE', 
                                     'age_of_onset':'AAO'}, inplace = True)

In [ ]:
#To tidy ancestry label for WGS samples
key["label"] = key["nba_label"].combine_first(key["wgs_label"])
key = key.drop(columns=["nba_label", "wgs_label"])
key

In [ ]:
#To tidy and create demographic subfiles from master key for each ancestry 
ancestries = {'AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH'}

for ancestry in ancestries:
    
    print(f'WORKING ON: {ancestry}')
    
    ## Subset to keep ancestry of interest 
    ancestry_key = key[key['label']==ancestry].copy()
    ancestry_key.reset_index(drop=True)
    
    # Convert phenotype to binary (1/2)
    ## Assign conditions so case=2 and controls=1, and -9 otherwise (matching PLINK convention)
    # PD = 2; control = 1
    pheno_mapping = {"PD": 2, "Control": 1}
    ancestry_key['PHENO'] = ancestry_key['phenotype'].map(pheno_mapping).astype('Int64')

    # Check value counts of pheno
    ancestry_key['PHENO'].value_counts(dropna=False)
    
    ## Get the PCs
    pcs = pd.read_csv(f'{REL11_PATH}/wgs/deepvariant_joint_calling/pcs/{ancestry}/{ancestry}_release11.eigenvec', sep='\t')
    
    #Select just first 5 PCs
    selected_columns = ['IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']
    pcs = pd.DataFrame(data=pcs.iloc[:, 1:7].values, columns=selected_columns)

    # Drop the first row (since it's now the column names)
    pcs = pcs.drop(0)

    # Reset the index to remove any potential issues
    pcs = pcs.reset_index(drop=True)
    
    # Check size
    print(f'PCs: {pcs.shape}')

    # Check value counts of SEX
    sex_og_values = ancestry_key['SEX'].value_counts(dropna=False)
    print(f'Sex value counts - original:\n {sex_og_values.to_string()}')
    
    # Convert sex to binary (1/2)
    ## Assign conditions so female=2 and men=1, and -9 otherwise (matching PLINK convention)
    # Female = 2; Male = 1
    sex_mapping = {"Female": 2, "Male": 1}
    ancestry_key['SEX'] = ancestry_key['SEX'].map(sex_mapping).astype('Int64')
    
    # Check value counts of SEX after recoding
    sex_recode_values = ancestry_key['SEX'].value_counts(dropna=False)
    print(f'Sex value counts - recoded:\n{sex_recode_values.to_string()}')
    
    ## Make covariate file
    df = pd.merge(ancestry_key,pcs, on='IID')
    print(f'Check columns for covariate file: {df.columns}')

    # Load information about related individuals in the ancestry analyzed
    related_df = pd.read_csv(f'{REL11_PATH}/meta_data/related_samples/{ancestry}_release11_vwb.related')
    print(f'Related individuals: {related_df.shape}')
    
    # Make a list of just one set of related people
    related_list = list(related_df['IID1'])
    related_list = [re.sub(r'_s.*$', '', iid) for iid in related_list]
    print(f'Number of related IIDs in dataset before filtering: {df["IID"].isin(related_list).sum()}')
    
    # Check value counts of related and remove only one related individual
    df = df[~df["IID"].isin(related_list)]

    # Check size
    print(f'Unrelated individuals: {df.shape}')
    
    #Make additional columns - FID, fatid and matid - these are needed for RVtests!!
    #RVtests needs the first 5 columns to be fid, iid, fatid, matid and sex otherwise it does not run correctly
    #Uppercase column name is ok
    #See https://zhanxw.github.io/rvtests/#phenotype-file
    df['FID'] = 0
    df['FATID'] = 0
    df['MATID'] = 0

    ## Clean up and keep columns we need 
    final_df = df[['FID','IID', 'FATID', 'MATID', 'SEX', 'AGE','AAO', 'PHENO','PC1', 'PC2', 'PC3', 'PC4', 'PC5']].copy()

    ##DO NOT replace missing values with -9 as this is misinterpreted by RVtests - needs to be nonnumeric
    #Leave missing values as NA
    
    #Check number of PD cases missing age
    pd_missAge = final_df[(final_df['PHENO']==2)&(final_df['AGE'].isna())]
    print(f'Number of PD cases missing age: {pd_missAge.shape[0]}')
    
    #Check number of controls missing age
    control_missAge = final_df[(final_df['PHENO']==1)&(final_df['AGE'].isna())]
    print(f'Number of controls missing age: {control_missAge.shape[0]}')

    ## Make file of sample IDs to keep 
    samples_toKeep = final_df[['FID', 'IID']].copy()
    samples_toKeep.columns = ['#FID','IID']
    
    samplestokeep_path = pathlib.Path(pathlib.Path.home(), f'/home/jupyter/workspace/ws_files/GCH1/WGS/{ancestry}/{ancestry}.samplestokeep')
    
    # Create the output CSV file's parent folder in the cloud storage bucket, if it doesn't already exist.
    if not samplestokeep_path.parent.exists():
        !mkdir -p {samplestokeep_path.parent}
        print(f'Created {samplestokeep_path.parent}')
    
    samples_toKeep.to_csv(samplestokeep_path, sep = '\t', index=False)

    finaldf_path = pathlib.Path(pathlib.Path.home(), f'/home/jupyter/workspace/ws_files/GCH1/WGS/{ancestry}/{ancestry}_covariate_file.txt')
    
    # Create the output CSV file's parent folder in the cloud storage bucket, if it doesn't already exist.
    if not finaldf_path.parent.exists():
        !mkdir -p {finaldf_path.parent}
        print(f'Created {finaldf_path.parent}')
    
    final_df.to_csv(finaldf_path, sep = '\t', na_rep='NA', index=False)

## Identification of GCH1 p.Ser80Asn variant carriers from WGS datasets

In [ ]:
#To make directories for each ancestry
ancestries = {'AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH'}

for ancestry in ancestries:
    !mkdir /home/jupyter/workspace/ws_files/GCH1/WGS/{ancestry}

In [ ]:
#To extract GCH1 p.Ser80Asn variant and variants surronding it using plink
for ancestry in ancestries:
    
    ! /home/jupyter/plink2 \
    --pfile ${REL11_PATH}/wgs/deepvariant_joint_calling/plink/{ancestry}/chr14_{ancestry}_release11 \
    --chr 14 \
    --from-bp 54902425 \
    --to-bp 54902925 \
    --make-pgen \
    --out /home/jupyter/workspace/ws_files/GCH1/WGS/{ancestry}/{ancestry}_GCH1

In [ ]:
#To tidy pfiles
for ancestry in ancestries:
    !sed -i '1!s/^[^\t]*/0/' /home/jupyter/workspace/ws_files/GCH1/WGS/{ancestry}/{ancestry}_GCH1.psam

In [ ]:
#To remove related samples
for ancestry in ancestries:
    ! /home/jupyter/plink2 \
    --pfile /home/jupyter/workspace/ws_files/GCH1/WGS/{ancestry}/{ancestry}_GCH1 \
    --keep /home/jupyter/workspace/ws_files/GCH1/WGS/{ancestry}/{ancestry}_covariate_file.txt \
    --chr 14 \
    --from-bp 54902425 \
    --to-bp 54902925 \
    --make-bfile \
    --out /home/jupyter/workspace/ws_files/GCH1/WGS/{ancestry}/{ancestry}_GCH1

In [ ]:
#To identify GCH1 p.Ser80Asn variant carriers
for ancestry in ancestries:
    ! /home/jupyter/plink1.9 \
    --bfile /home/jupyter/workspace/ws_files/GCH1/WGS/{ancestry}/{ancestry}_GCH1 \
    --recode A \
    --out /home/jupyter/workspace/ws_files/GCH1/WGS/{ancestry}/{ancestry}_GCH1

## Haplotype analysis of GCH1 p.Ser80Asn carrier in EAS cohort

In [ ]:
%%bash

#convert plink pfile to vcf for phasing
#extract region within 1MB of the GCH1 p.Ser80Asn variant

/home/jupyter/plink2 --pfile /home/jupyter/workspace/ws_files/GCH1/WGS/EAS/GCH1_EAS \
                     --chr 14 \
                     --mind 0.01 \
                     --from-bp 54402425 \
                     --to-bp 55402425 \
                     --set-all-var-ids @:#:$r:$a \
                     --rm-dup exclude-mismatch \
                     --snps-only just-acgt \
                     --export vcf id-paste=iid \
                     --out /home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14

In [ ]:
%%bash 

bgzip -c /home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14.vcf > /home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14.vcf.gz
tabix -p vcf -f /home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14.vcf.gz

In [ ]:
%%bash

java -jar /home/jupyter/workspace/beagle/beagle.29Oct24.c8e.jar \
	gt=/home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14.vcf.gz \
	map=/home/jupyter/workspace/beagle/no_chr_in_chrom_field/plink.chr14.GRCh38.map \
	out=/home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14_phased

In [ ]:
%%bash

java -jar /home/jupyter/workspace/beagle/hap-ibd.jar \
        gt=/home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14_phased.vcf.gz \
        map=/home/jupyter/workspace/beagle/no_chr_in_chrom_field/plink.chr14.GRCh38.map \
        out=/home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14_ibd

In [ ]:
!ls -lh /home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14_ibd.ibd.gz

In [ ]:
ibd=pd.read_csv('/home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14_ibd.ibd.gz',compression='gzip',sep='\t', header=None,names=['sample1','hap1','sample2','hap2','chrom','Start','End','cM_len'])
ibd

In [ ]:
# get carrier list
carrier_list = pd.read_csv('/home/jupyter/workspace/ws_files/GCH1_carrier.txt')

# read ibd results
ibd=pd.read_csv('/home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14_ibd.ibd.gz',compression='gzip',sep='\t', header=None,names=['sample1','hap1','sample2','hap2','chrom','Start','End','cM_len'])

# subset just ibd segement among the carriers
keep_ibd=ibd.loc[(ibd['sample1'].isin(carrier_list['IID'])) & (ibd['sample2'].isin(carrier_list['IID']))]

keep_ibd.to_csv('/home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14_ibd_sharing.csv',index=False)

In [ ]:
%%R

library(ggplot2)
library(dplyr)
library(RColorBrewer)


data=read.csv('/home/jupyter/workspace/ws_files/GCH1/WGS/EAS/EAS_chr14_ibd_sharing.csv')

data <- data %>%
  arrange(FAM) %>%  # First sort by val. This sort the dataframe but NOT the factor levels
  mutate(SegmentID = row_number())%>%
  mutate(SegmentLength = End - Start) %>%
  arrange(FAM, desc(SegmentLength))

median(data$cM_len)
min(data$cM_len)
max(data$cM_len)

# Plot the IBD segments
ggplot(data, aes(x = Start, xend = End, y = SegmentID, yend = SegmentID, color = FAM)) +
  geom_segment(size = 1) +
  geom_text(aes(x = 1, label = Pair), hjust = 0.5, color = "black", size = 5) +  # Fixed y position for all text
  facet_wrap(~Chrom, scales = "free_x") +
  labs(title = "Overlapping IBD Segments",
       x = "Genomic Position", y = "Segment ID") +
  theme_minimal() +
  theme(panel.border = element_rect(color = "black", fill = NA, linewidth = 1.5),  # Add black border
        axis.title.y=element_blank(),
        axis.text.y = element_blank(),  # Remove Y-axis ticks
        axis.ticks.y = element_blank(),  # Remove Y-axis ticks
        strip.background = element_blank(),  # Remove facet strip background
        strip.text = element_blank(),  # Remove facet strip text
        panel.grid.major = element_blank(), panel.grid.minor = element_blank(),panel.background = element_blank())+
  geom_vline(xintercept = 40322386, color = "dark grey", size = 0.5,linetype="dashed")+
  scale_color_manual(values = c("UR" = "steelblue", "GP2-FAM-1" = "#FC4E07"))+
  labs(title = "title") +
  labs(title = NULL)